In [1]:
import kaleidocell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
from datetime import datetime

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

In [2]:
OUT_DIR = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/"
os.makedirs(OUT_DIR, exist_ok=True)



In [3]:
checkpoint("Loading results_mp")
results_mp = kaleidocell.load("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/04_NMF_GSEAonHGNC/04_3_nmf_all_gsea_c7/results_mp.kc")
checkpoint(f"Loaded: {len(results_mp['mp_dict'])} MPs")

[2026-08-04 09:33:33] Loading results_mp
[2026-08-04 09:33:35] Loaded: 30 MPs


In [4]:
mp_dict = results_mp["mp_dict"]


In [5]:
checkpoint("Loading summary/metrics")
summary = results_mp["metrics"]
print(summary.columns.tolist())

checkpoint("Computing max_gene_score per MP")
summary["max_gene_score"] = pd.Series({mp: genes.max() for mp, genes in mp_dict.items()})

checkpoint("Loading meaningful_mps and computing n_pathways")
top_gsea_per_mp = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/05_significant_MP/05_all_immune_cells/05_1_gsea_c7_immunesigdb_enrichr.csv")

n_pathways_per_mp = top_gsea_per_mp.groupby("MP")["Term"].nunique()
summary["n_pathways"] = summary.index.map(n_pathways_per_mp).fillna(0)

meaningful_mps = top_gsea_per_mp["MP"].unique()

[2026-08-04 09:33:35] Loading summary/metrics
['sampleCoverage', 'silhouette', 'meanSimilarity', 'nPrograms', 'nGenes']
[2026-08-04 09:33:35] Computing max_gene_score per MP
[2026-08-04 09:33:35] Loading meaningful_mps and computing n_pathways


In [45]:
metrics_higher_better = ["silhouette", "max_gene_score", "n_pathways", "sampleCoverage", "meanSimilarity"]

flag_cols = {}
for col in metrics_higher_better:
    if col == "n_pathways":
        flag_cols[col] = summary[col] == 0
    else:
        flag_cols[col] = pd.Series(False, index=summary.index)

mt_fraction = {mp: sum(g.startswith("MT-") for g in mp_dict[mp].index) / len(mp_dict[mp]) for mp in summary.index}
summary["mt_fraction"] = pd.Series(mt_fraction)
n_genes = {mp: len(mp_dict[mp]) for mp in summary.index}
summary["n_genes"] = pd.Series(n_genes)

core_metrics = ["silhouette", "sampleCoverage", "meanSimilarity"]

ranks = summary[core_metrics].copy()
ranks[["sampleCoverage", "silhouette", "meanSimilarity"]] = summary[["sampleCoverage", "silhouette", "meanSimilarity"]].rank(ascending=False)

summary["combined_rank"] = ranks.mean(axis=1)

display_cols = core_metrics + ["max_gene_score", "n_pathways", "n_genes", "mt_fraction", "combined_rank"]
table_df = summary[display_cols].round(3)

norm_df = pd.DataFrame(index=table_df.index, columns=table_df.columns, dtype=float)

sim_max_reference = 0.6

for col in metrics_higher_better:
    if col == "silhouette":
        clipped = table_df[col].clip(lower=0)
        col_max = clipped.max()
        norm_df[col] = clipped / col_max if col_max > 0 else 0.0
    elif col == "sampleCoverage":
        col_range = table_df[col].max() - table_df[col].min()
        norm_df[col] = 1 - (table_df[col] - table_df[col].min()) / col_range if col_range > 0 else 0.5
    elif col == "meanSimilarity":
        norm_df[col] = (table_df[col] - table_df[col].min()) / (table_df[col].max() - table_df[col].min())
    else:
        col_range = table_df[col].max() - table_df[col].min()
        norm_df[col] = (table_df[col] - table_df[col].min()) / col_range if col_range > 0 else 0.5

mt_threshold = 0.3

meansim_threshold = 0.2

norm_df["mt_fraction"] = (table_df["mt_fraction"] / mt_threshold).clip(upper=1.0)
norm_df["combined_rank"] = 1 - (table_df["combined_rank"] - table_df["combined_rank"].min()) / (table_df["combined_rank"].max() - table_df["combined_rank"].min())
cov_col = table_df["sampleCoverage"]
norm_df["sampleCoverage"] = (cov_col - cov_col.min()) / (cov_col.max() - cov_col.min())
col_max = table_df["meanSimilarity"].max()
norm_df["meanSimilarity"] = np.where(
    table_df["meanSimilarity"] < meansim_threshold, 0.0, (table_df["meanSimilarity"] - meansim_threshold) / (col_max - meansim_threshold))
gene_range = table_df["n_genes"].max() - table_df["n_genes"].min()
norm_df["n_genes"] = (table_df["n_genes"] - table_df["n_genes"].min()) / gene_range if gene_range > 0 else 0.5

In [48]:
print(ranks["sampleCoverage"].head())
print(summary[["sampleCoverage", "combined_rank"]].sort_values("sampleCoverage", ascending=False).head())
print(ranks["sampleCoverage"].sort_values().head())

MP1    24.0
MP2     2.0
MP3     7.0
MP4    22.0
MP5    15.0
Name: sampleCoverage, dtype: float64
      sampleCoverage  combined_rank
MP26        0.611111      19.333333
MP2         0.495726       2.000000
MP14        0.487179      12.666667
MP6         0.478632      16.333333
MP29        0.435897      20.666667
MP26    1.0
MP2     2.0
MP14    3.0
MP6     4.0
MP29    5.0
Name: sampleCoverage, dtype: float64


In [ ]:
INFERNO_PINK = "#ed1a72"
HEADER_COLOR = "#63042c"
YELLOW = "#ffd23f" # not used
INDEX_COLOR = "#fdece0"
OUTLIER_COLOR = "#f57b17"  # or #82042a
cmap = mcolors.LinearSegmentedColormap.from_list("pastel_to_pink", ["#ffffff", INFERNO_PINK])

fig, ax = plt.subplots(figsize=(12, max(4, len(table_df) * 0.4)))
ax.axis("off")

int_cols = ["n_pathways", "n_genes"]
float_cols = [c for c in display_cols if c not in int_cols]

table_df = summary[display_cols].copy()

cell_text = table_df.copy()

for col in float_cols:
    cell_text[col] = table_df[col].apply(lambda x: f"{x:.5f}")

for col in int_cols:
    cell_text[col] = table_df[col].apply(lambda x: f"{int(round(x))}")

tbl = ax.table(
    cellText=cell_text.values,
    rowLabels=table_df.index,
    colLabels=table_df.columns,
    cellLoc="center",
    loc="center"
)

for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor(HEADER_COLOR)
        cell.set_text_props(color="white", weight="bold")
    elif col == -1:
        cell.set_facecolor(INDEX_COLOR)
        cell.set_text_props(weight="bold")
    else:
        col_name = table_df.columns[col]
        mp_name = table_df.index[row - 1]
        val = norm_df.loc[mp_name, col_name]
        cell.set_facecolor(cmap(val))
        cell.set_text_props(color="white" if val > 0.6 else "black")

        if col_name == "mt_fraction" and table_df.loc[mp_name, col_name] > 0.3:
            cell.set_facecolor(OUTLIER_COLOR)
            cell.set_text_props(color="white", weight="bold")
        if col_name == "n_pathways" and table_df.loc[mp_name, col_name] == 0:
            cell.set_facecolor(OUTLIER_COLOR)
            cell.set_text_props(color="white", weight="bold")
        if col_name == "n_genes" and (table_df.loc[mp_name, col_name] < 15 or table_df.loc[mp_name, col_name] > 1000):
            cell.set_facecolor(OUTLIER_COLOR)
            cell.set_text_props(color="white", weight="bold")

tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.5)

plt.title("MP Quality Metrics Overview", fontsize=14, fontweight="bold", y=0.93)
plt.savefig(os.path.join(OUT_DIR, "05_11_mp_quality_overview_table2.pdf"), bbox_inches="tight")
plt.close()